# B-05: arcsinh target transform on the enriched trees (D-42)

Same enriched-feature tree pipeline as **B-03** (v2 hourly + daily, tuned daily HPO), but the trees are fit on
**`y' = arcsinh(FCH4)`** and predictions are back-transformed with **`sinh`** before evaluation (metrics in
original nmol space, on observed days). `arcsinh` is chosen over `log` because CH4 flux is **signed** (range
≈ −1559…+6161): it is ~linear near 0 and ~log in the tails, compressing the episodic spikes that dominate
squared error and depress R². Back-transform uses **Duan smearing** (sinh over sampled training residuals) to
remove the Jensen bias of a naive `sinh`. Baselines (persistence/climatology) are untransformed; metrics in
original nmol space on observed days. Final cells compare **B-05 vs B-03 vs FC-01**.

**Result preview (negative):** even bias-corrected, the transform does **not** beat the identity-target trees
(B-03) on original-space R² — compressing the spikes trades spike accuracy (which dominates the variance) for
bulk accuracy. Logged as a documented negative result; B-03 remains the production config.

In [1]:
from pathlib import Path
import sys, datetime, numpy as np, pandas as pd
sys.path.insert(0, "../../src")
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from evaluation.metrics import full_metrics
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
ALGOS=["RF","XGB"]
HORIZONS={"A":[1,6,12,24,48],"B":[1,3,7,14]}
T2_FOLDS=[("2018-06-30","2018-07-01","2018-12-31"),("2018-12-31","2019-01-01","2019-05-31")]
DUM=["is_t2","is_t4","is_t9"]
TF=np.arcsinh; ITF=np.sinh   # target transform / inverse

matv2=pd.read_csv(HOURLY/"forecast_features_v2.csv",low_memory=False)
matv2["Datetime"]=pd.to_datetime(matv2["Datetime"],format="mixed")
AR_A=[c for c in matv2.columns if c.startswith("ar_")]
FX_A=[c for c in matv2.columns if c.startswith("fx")]
print("hourly v2 rows",len(matv2),"| AR_A",len(AR_A),"| FX_A",len(FX_A))

hourly v2 rows 210459 | AR_A 20 | FX_A 26


## 1  Helpers (trees fit on arcsinh target; baselines untransformed)

In [2]:
def fit_model(algo, tr, feat, track="A"):
    imp=SimpleImputer(strategy="mean"); Xi=imp.fit_transform(tr[feat].values)
    if algo=="RF":
        p=dict(min_samples_leaf=5) if track=="A" else dict(min_samples_leaf=10,max_features=0.5)
        m=RandomForestRegressor(n_estimators=500,n_jobs=-1,random_state=42,**p)
    else:
        p=dict(max_depth=6,learning_rate=0.05,n_estimators=500) if track=="A" \
          else dict(max_depth=2,learning_rate=0.02,n_estimators=400,min_child_weight=10)
        m=XGBRegressor(subsample=0.8,colsample_bytree=0.8,n_jobs=-1,random_state=42,**p)
    z=TF(tr["target"].values); m.fit(Xi, z)                    # <-- fit on arcsinh(y)
    rq=np.quantile(z - m.predict(Xi), np.linspace(0.01,0.99,256))  # Duan smearing residual sample
    return m,imp,rq

def predict(m, imp, rq, X):
    zt=m.predict(imp.transform(X))                              # transformed-space prediction
    return ITF(zt[:,None]+rq[None,:]).mean(axis=1)             # smearing-corrected back-transform

def rmse(y,p): return float(np.sqrt(mean_squared_error(y,p)))
def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r2=r2_score(y,p) if (len(y)>1 and np.var(y)>0) else np.nan
    return rmse(y,p), float(mean_absolute_error(y,p)), r2, float(np.mean(p-y))

def climatology(tr, te, unit):
    t=te["tower"].iloc[0]; trt=tr[tr.tower==t]
    if len(trt)<24: trt=tr
    gl=float(trt["target"].mean())
    if unit=="h":
        g=trt.assign(mo=trt.ttime.dt.month,hr=trt.ttime.dt.hour).groupby(["mo","hr"])["target"].mean()
        keys=list(zip(te.ttime.dt.month,te.ttime.dt.hour)); return np.array([g.get(k,gl) for k in keys])
    g=trt.assign(mo=trt.ttime.dt.month).groupby("mo")["target"].mean()
    return np.array([g.get(m,gl) for m in te.ttime.dt.month])

def build_hourly_tables():
    T={t: matv2[matv2.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_A, FX_A

def build_daily_tables():
    dv=pd.read_csv(HOURLY/"forecast_daily_v2.csv",low_memory=False)
    dv["Datetime"]=pd.to_datetime(dv["Datetime"],format="mixed")
    AR_B=[c for c in dv.columns if c.startswith("ar_")]; FX_B=[c for c in dv.columns if c.startswith("fx")]
    T={t: dv[dv.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_B, FX_B

def aligned(T, h, unit, ar_cols, fx_cols):
    parts=[]; off=pd.Timedelta(hours=h) if unit=="h" else pd.Timedelta(days=h)
    for t,df in T.items():
        f=df[ar_cols+DUM].copy()
        for c in fx_cols: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["persist"]=df["y_gapfilled"]
        f["tower"]=t; f["ttime"]=df.index+off
        parts.append(f)
    return pd.concat(parts)

## 2  Run one horizon

In [3]:
def emit(track,h,t,bydict):
    rp=rmse(*bydict["persist"]); rc=rmse(*bydict["clim"]); rows=[]
    y_naive=bydict["persist"][1]
    for model,(y,p) in bydict.items():
        r,mae,r2,mbe=metrics(y,p)
        fm=full_metrics(y,p,y_naive=y_naive)
        rows.append(dict(track=track,horizon=h,tower=t,model=model,RMSE=round(r,3),MAE=round(mae,3),
            R2=(round(r2,3) if np.isfinite(r2) else np.nan),MBE=round(mbe,3),n_test=len(y),
            skill_persist=round(1-r/rp,3) if rp>0 else np.nan,
            skill_clim=round(1-r/rc,3) if rc>0 else np.nan,
            WAPE=round(fm["WAPE"],4) if np.isfinite(fm["WAPE"]) else np.nan,
            MASE=round(fm["MASE"],4) if np.isfinite(fm["MASE"]) else np.nan,
            sMAPE=round(fm["sMAPE"],4) if np.isfinite(fm["sMAPE"]) else np.nan,
            MAPE=round(fm["MAPE"],4) if np.isfinite(fm["MAPE"]) else np.nan,
            MAPE_n_excluded=fm["MAPE_n_excluded"]))
    return rows

def run_horizon(track, h, unit, ar_cols, fx_cols, T):
    feat=ar_cols+fx_cols+DUM; A=aligned(T,h,unit,ar_cols,fx_cols); rows=[]
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    fitted={a:fit_model(a,tr,feat,track) for a in ALGOS}
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
        if len(te)<10: continue
        bd={"persist":(te["target"].values,te["persist"].values),
            "clim":(te["target"].values,climatology(tr,te,unit))}
        for a in ALGOS:
            m,imp,rq=fitted[a]; bd[a]=(te["target"].values,predict(m,imp,rq,te[feat].values))
        rows+=emit(track,h,t,bd)
    acc={k:([],[]) for k in ["persist","clim"]+ALGOS}
    for cut,s,e in T2_FOLDS:
        trf=A[(A.ttime<=pd.Timestamp(cut)) & A.target.notna()]
        te=A[(A.tower==2)&(A.ttime>=pd.Timestamp(s))&(A.ttime<=pd.Timestamp(e))&A.target.notna()]
        if len(te)<5 or len(trf)<50: continue
        y=te["target"].values
        acc["persist"][0].append(y); acc["persist"][1].append(te["persist"].values)
        acc["clim"][0].append(y); acc["clim"][1].append(climatology(trf,te,unit))
        for a in ALGOS:
            m,imp,rq=fit_model(a,trf,feat,track); acc[a][0].append(y); acc[a][1].append(predict(m,imp,rq,te[feat].values))
    if acc["persist"][0]:
        bd={k:(np.concatenate(v[0]),np.concatenate(v[1])) for k,v in acc.items()}
        rows+=emit(track,h,2,bd)
    return rows

## 3  Run both tracks

In [4]:
TA,AR_A_,FX_A_=build_hourly_tables(); TB,AR_B_,FX_B_=build_daily_tables()
ALL=[]
for h in HORIZONS["A"]:
    ALL+=run_horizon("A",h,"h",AR_A_,FX_A_,TA); print("Track A h=",h,"done",flush=True)
for h in HORIZONS["B"]:
    ALL+=run_horizon("B",h,"D",AR_B_,FX_B_,TB); print("Track B d=",h,"done",flush=True)
R=pd.DataFrame(ALL); print("rows",len(R))

Track A h= 1 done


Track A h= 6 done


Track A h= 12 done


Track A h= 24 done


Track A h= 48 done


Track B d= 1 done


Track B d= 3 done


Track B d= 7 done


Track B d= 14 done


rows 108


## 4  Results + save b05_summary.csv

In [5]:
pd.set_option("display.width",200)
print("=== Track B (daily) — R2 (towers 4/9) ===")
print(R[(R.track=="B")&(R.tower.isin([4,9]))&(R.model.isin(["RF","XGB"]))]
      .pivot_table(index=["tower","model"],columns="horizon",values="R2").round(3).to_string())
print("\n=== daily MBE (back-transform bias check, RF/XGB) ===")
print(R[(R.track=="B")&(R.tower.isin([4,9]))&(R.model.isin(["RF","XGB"]))]
      .pivot_table(index=["tower","model"],columns="horizon",values="MBE").round(2).to_string())
R.to_csv(RESULTS/"b05_summary.csv",index=False); print("\nsaved results/b05_summary.csv")

=== Track B (daily) — R2 (towers 4/9) ===
horizon         1      3      7      14
tower model                            
4     RF     0.332  0.312  0.277  0.281
      XGB    0.337  0.327  0.287  0.320
9     RF     0.342  0.322  0.347  0.331
      XGB    0.266  0.269  0.297  0.293

=== daily MBE (back-transform bias check, RF/XGB) ===
horizon         1     3     7     14
tower model                         
4     RF     -2.98 -0.67 -0.68 -1.01
      XGB    10.17  9.49  8.21  8.30
9     RF     -2.98 -4.44 -3.59 -5.19
      XGB     1.51 -0.14 -0.28 -1.77

saved results/b05_summary.csv


## 5  Compare B05 (arcsinh) vs B03 (identity) vs FC-01

In [6]:
b03=pd.read_csv(RESULTS/"b03_summary.csv"); fc01=pd.read_csv(RESULTS/"fc01_summary.csv")
key=["track","tower","horizon","model"]
j=R.merge(b03[key+["R2"]],on=key,suffixes=("_b05","_b03")).merge(
    fc01[key+["R2"]].rename(columns={"R2":"R2_fc01"}),on=key)
j["d_vs_b03"]=j["R2_b05"]-j["R2_b03"]
print("=== mean R2 delta B05-B03 by track/model (towers 4/9) ===")
print(j[j.tower.isin([4,9])].groupby(["track","model"])[["d_vs_b03"]].mean().round(3).to_string())
print("\n=== daily h=1 R2: FC-01 -> B03 -> B05 (towers 4/9) ===")
print(j[(j.track=="B")&(j.horizon==1)&(j.tower.isin([4,9]))][["tower","model","R2_fc01","R2_b03","R2_b05"]].round(3).to_string(index=False))
print("\n=== best daily R2 per tower: FC-01 / B03 / B05 ===")
for t in [4,9]:
    a=fc01[(fc01.track=="B")&(fc01.tower==t)&(fc01.model.isin(["RF","XGB"]))]["R2"].max()
    b=b03[(b03.track=="B")&(b03.tower==t)&(b03.model.isin(["RF","XGB"]))]["R2"].max()
    c=R[(R.track=="B")&(R.tower==t)&(R.model.isin(["RF","XGB"]))]["R2"].max()
    print(f"  Tower {t}:  FC-01 {a:.3f}  ->  B03 {b:.3f}  ->  B05 {c:.3f}")

=== mean R2 delta B05-B03 by track/model (towers 4/9) ===
               d_vs_b03
track model            
A     RF         -0.073
      XGB        -0.041
      clim        0.000
      persist     0.000
B     RF         -0.020
      XGB         0.001
      clim        0.000
      persist     0.000

=== daily h=1 R2: FC-01 -> B03 -> B05 (towers 4/9) ===
 tower   model  R2_fc01  R2_b03  R2_b05
     4 persist    0.440   0.440   0.440
     4    clim   -0.022  -0.022  -0.022
     4      RF    0.256   0.357   0.332
     4     XGB    0.263   0.362   0.337
     9 persist    0.231   0.231   0.231
     9    clim    0.045   0.045   0.045
     9      RF    0.304   0.388   0.342
     9     XGB    0.189   0.324   0.266

=== best daily R2 per tower: FC-01 / B03 / B05 ===
  Tower 4:  FC-01 0.263  ->  B03 0.362  ->  B05 0.337
  Tower 9:  FC-01 0.304  ->  B03 0.388  ->  B05 0.347


## 6  Append to benchmarks.csv (B05)

In [7]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="B05"]
rows=[]
for _,r in R.iterrows():
    rows.append({"replication":"B05","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":"enriched_v2 + arcsinh target transform","track":r["track"],"horizon":int(r["horizon"]),
        "split":"fc_main" if r.tower in (4,9) else "fc_t2folds","R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],
        "MBE":r["MBE"],"n_test":int(r["n_test"]),"skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],
        "WAPE":r["WAPE"],"MASE":r["MASE"],"sMAPE":r["sMAPE"],"MAPE":r["MAPE"],"MAPE_n_excluded":r["MAPE_n_excluded"],
        "date":today,"notes":"B05 arcsinh target transform on enriched trees (Duan-smearing back-transform); NEGATIVE result - does not beat B03 in original-space R2 (D-42); WAPE/MASE/sMAPE/MAPE added D-44b"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} B05 rows. Total {len(comb)}.")

Wrote 108 B05 rows. Total 3665.
